In [2]:
##OLAP (Online Anlytical Process)

#It is used for model training.
#It deals with historical data.
#It is used for complex queries like SUM,AVG etc.
#It is slower than OLTP.
#There is no such concept of ACID property.
#OLTP ------------> OLAP can be converted using ETL
#OLAP is column major.


#OLAP system will collect historical data may be using OLTP
# -----> then it Aggregate it
# -----> show results in reports/graphs.




In [3]:
#Difference Between OLTP AND OLAP
#    Features     |   OLTP         |         OLAP
#                 |                | 
#      Purpose    | Transaction    |    Analysis
#       Data      | Current        |    Historical
#       Queries   | Simple         |    Complex
#       Speed     | Very Fast      |     Slower   
#       Users     | Customer,clerks|    Manager,analyst

In [4]:
import sqlite3
import pandas as pd
import numpy as np
from datetime import datetime,timedelta


In [5]:
conn=sqlite3.connect('unisole_analytics.db')
cursor=conn.cursor()


In [6]:
cursor.execute('''  
CREATE TABLE IF NOT EXISTS
historical_rides(
               ride_id INTEGER,
               user_id TEXT ,
               distance REAL ,     
               ride_time_minutes INTEGER,
               rating INTEGER,
               date DATE
)               
''')

data=[]   #creating empty list
start_date=datetime(2025,1,1)     #international format is (yyyy-mm-dd)
for i in range(10000):
    data.append((i,f"USER_{np.random.randint(1,500)}",
                 round(np.random.uniform(2,50),2),
                 round(np.random.uniform(100,1000),2),
                 np.random.randint(10,120),
                 np.random.randint(1,6),
                 (start_date +
                  timedelta(days=np.random.randint(1,365))).strftime('%y-%m-%d')
             ))
cursor.executemany(
        "INSERT INTO historical_rides VALUES (?,?,?,?,?,?,?)",
        data
        )     
conn.commit()
print("OLAP table populated with 10,000 historical records")

OLAP table populated with 10,000 historical records


In [7]:
#monthly revenue analyis.
query=""" 
SELECT 
  strftime("%m",date) AS Month,
  COUNT(ride_id) AS Total_Rides,
  SUM(price) AS Total_revenue,
  AVG(rating) AS Avg_Rating
  FROM historical_rides
GROUP BY Month
ORDER BY Month
"""
monthly_report=pd.read_sql_query(query,conn)
print("monthly business insights")
print("monthly_report")

monthly business insights
monthly_report


In [8]:
# ML Feature Correlation: Distance vs Price

df_ml = pd.read_sql_query(
    "SELECT distance, price FROM historical_rides",
    conn
)

correlation = df_ml['distance'].corr(df_ml['price'])

print(f"Correlation between Distance and Price: {correlation:.2f}")

# Industry Note:
# Agar ye millions of rows hote,
# toh hum Parquet file use karte (Column-major)

Correlation between Distance and Price: 0.00


In [9]:
query = """
SELECT
    user_id,
    SUM(price) AS lifetime_value,
    RANK() OVER (
        ORDER BY SUM(price) DESC
    ) AS user_rank
FROM historical_rides
GROUP BY user_id
LIMIT 10;
"""

rankings = pd.read_sql_query(query, conn)

print("--- Top 10 High-Value Users (LTV) ---")
print(rankings)

--- Top 10 High-Value Users (LTV) ---
    user_id  lifetime_value  user_rank
0  USER_226       112239.11          1
1   USER_55       108410.88          2
2  USER_480       108268.02          3
3  USER_322       108230.37          4
4  USER_366       107923.15          5
5  USER_376       107014.59          6
6   USER_17       105482.90          7
7  USER_133       104458.84          8
8  USER_220       104365.73          9
9  USER_311       103859.64         10
